In [34]:
import pandas as pd

### what columns do we have

In [35]:
data = pd.read_csv('../../data/task_2_data_ex.csv')
data.head()

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10


### analyse the numeric values in the dataset
we can see that we have 2024 year with up to 12 month (not as 2025 year and 0 month)

In [36]:
data.describe()

,year,month,produced_material,produced_material_production_type,component_material,component_material_production_type
count,1320.0,1320.000000,1320.000000,1320.000000,1320.000000,480.000000
mean,2024.0,6.500000,65479.954545,8002.818182,81841.136364,8002.500000
std,0.0,3.453361,21916.019385,2.657669,11933.718591,2.695392
min,2024.0,1.000000,10000.000000,8000.000000,50000.000000,8000.000000
25%,2024.0,3.750000,50005.000000,8001.000000,80007.000000,8000.750000
50%,2024.0,6.500000,80007.000000,8002.000000,90004.500000,8001.500000
75%,2024.0,9.250000,80070.000000,8007.000000,90027.000000,8003.250000
max,2024.0,12.000000,80079.000000,8007.000000,90050.000000,8007.000000


### we have a lot of NaN data in ```component_material_production_type``` feature

because of Raw Materials (RM) and Additives (ADD). they don't have any components (they are leaves in the hierarchy tree)

In [37]:
data.isnull().sum()

year                                    0
month                                   0
produced_material                       0
produced_material_production_type       0
produced_material_release_type          0
produced_material_quantity              0
component_material                      0
component_material_production_type    840
component_material_release_type         0
component_material_quantity             0
plant_id                                0
dtype: int64

### we can see that `produced_material_quantity` and `component_material_quantity` has `str` type (`not int`)

In [38]:
data.dtypes

year                                    int64
month                                   int64
produced_material                       int64
produced_material_production_type       int64
produced_material_release_type            str
produced_material_quantity                str
component_material                      int64
component_material_production_type    float64
component_material_release_type           str
component_material_quantity               str
plant_id                                  str
dtype: object

# Business logic
1. extract data from csv file and process float nums into valid values
2. concatinate all necessary time to year to create FIN (aggregate from monthly to annualy)
3. building indexing (like B+ tree) to speed up the search
4. for each FIN and PROD create a new hierarchy raw
5. final report of created new DataFrame

In [39]:
EXPLODEABLE = {"FIN", "PROD"}  # ADD and RM are leaves

FINAL_COLUMNS = [
    "plant",
    "fin_material_id",
    "fin_material_release_type",
    "fin_material_production_type",
    "fin_production_quantity",
    "prod_material_id",
    "prod_material_release_type",
    "prod_material_production_type",
    "prod_material_production_quantity",
    "component_id",
    "component_material_release_type",
    "component_material_production_type",
    "component_consumption_quantity",
    "year",
]

In [40]:
from pathlib import Path

def load_and_clean_bom(path: str) -> pd.DataFrame:
    path = Path(path)

    id_dtypes = {
        "produced_material": "string",
        "component_material": "string",
        "produced_material_release_type": "string",
        "component_material_release_type": "string",
        "plant_id": "string",
    }

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, dtype=id_dtypes)
    elif path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path, dtype=id_dtypes)
    else:
        raise ValueError(f"Unsupported file format: {path.suffix}")

    quantity_columns = [
        "produced_material_quantity",
        "component_material_quantity",
    ]

    for column in quantity_columns:
        df[column] = pd.to_numeric(
            df[column]
            .astype("string")
            .str.replace(",", "", regex=False),
            errors="raise",
        )

    production_type_columns = [
        "produced_material_production_type",
        "component_material_production_type",
    ]

    for column in production_type_columns:
        df[column] = pd.to_numeric(
            df[column], errors="coerce"
        ).astype("Int64")

    df["year"] = df["year"].astype("Int64")
    df["month"] = df["month"].astype("Int64")

    return df

In [41]:
def aggregate_bom_to_year(df: pd.DataFrame) -> pd.DataFrame:
    """One BoM edge per plant/year/material/component, quantities summed."""
    group_cols = [
        "plant_id", "year",
        "produced_material", "produced_material_production_type", "produced_material_release_type",
        "component_material", "component_material_production_type", "component_material_release_type",
    ]

    return (
        df.groupby(group_cols, dropna=False, as_index=False)
        .agg(
            produced_material_quantity=("produced_material_quantity", "sum"),
            component_material_quantity=("component_material_quantity", "sum"),
        )
    )

In [42]:
def build_bom_index(df: pd.DataFrame) -> dict[tuple, pd.DataFrame]:
    """(plant, year, produced_material) -> rows of its components."""
    return {
        key: group.reset_index(drop=True)
        for key, group in df.groupby(["plant_id", "year", "produced_material"], sort=False)
    }

In [43]:
from collections import deque

def explode_fin_material(
    bom_index: dict,
    plant: str,
    year: int,
    fin_material: str,
    fin_meta: pd.Series,
) -> list[dict]:
    rows = []
    seen = set()

    fin_edges = bom_index.get((plant, year, fin_material), pd.DataFrame())

    # Start below FIN; do not output FIN itself as prod_material_id
    queue = deque(
        edge["component_material"]
        for _, edge in fin_edges.iterrows()
        if edge["component_material_release_type"] in EXPLODEABLE
    )

    while queue:
        material = queue.popleft()

        if material in seen:
            continue
        seen.add(material)

        edges = bom_index.get((plant, year, material), pd.DataFrame())

        for _, edge in edges.iterrows():
            rows.append({
                "plant": plant,
                "fin_material_id": fin_material,
                "fin_material_release_type": fin_meta["produced_material_release_type"],
                "fin_material_production_type": fin_meta["produced_material_production_type"],
                "fin_production_quantity": fin_meta["produced_material_quantity"],
                "prod_material_id": edge["produced_material"],
                "prod_material_release_type": edge["produced_material_release_type"],
                "prod_material_production_type": edge["produced_material_production_type"],
                "prod_material_production_quantity": edge["produced_material_quantity"],
                "component_id": edge["component_material"],
                "component_material_release_type": edge["component_material_release_type"],
                "component_material_production_type": edge["component_material_production_type"],
                "component_consumption_quantity": edge["component_material_quantity"],
                "year": year,
            })

            if edge["component_material_release_type"] in EXPLODEABLE:
                queue.append(edge["component_material"])

    return rows

In [44]:
def explode_all(bom_annual: pd.DataFrame) -> pd.DataFrame:
    bom_index = build_bom_index(bom_annual)

    fin_roots = (
        bom_annual[bom_annual["produced_material_release_type"] == "FIN"]
        .drop_duplicates(["plant_id", "year", "produced_material"])
    )

    all_rows = []
    for _, fin in fin_roots.iterrows():
        all_rows.extend(
            explode_fin_material(
                bom_index,
                fin["plant_id"],
                fin["year"],
                fin["produced_material"],
                fin,
            )
        )

    return pd.DataFrame(all_rows, columns=FINAL_COLUMNS)

In [45]:
def run_pipeline(path: str) -> pd.DataFrame:
    data = load_and_clean_bom(path)
    bom_annual = aggregate_bom_to_year(data)
    exploded = explode_all(bom_annual)
    return exploded

In [46]:
run_pipeline(path="../../data/task_2_data_ex.csv").head(10)

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007,11303.0,2024
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,<NA>,598.0,2024
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,<NA>,242.0,2024
3,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001,41769.0,2024
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,<NA>,355.0,2024
5,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90003,ADD,<NA>,121.0,2024
6,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,80000,PROD,8000,23360.0,2024
7,RLT_10,10000,FIN,8002,11708.0,80010,PROD,8001,21013.0,90004,ADD,<NA>,1242.0,2024
8,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,<NA>,30498.0,2024
9,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,90005,ADD,<NA>,829.0,2024


In [47]:
data.head()

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10
